# Notebook 08 — full assertion-claim review correction

This notebook resumes from the verified, published Stage 08 pilot evidence and rebuilds its human review under Project Pipeline §§13.4 and 20.7.

1. Create or import this notebook on Kaggle. Attach the exact published **Stage 05, Stage 06, Stage 07 and original Stage 08** datasets for this run. Each input needs its external stage manifest and either the original ZIP or its expanded contents. Use CPU.
2. Run **Cells 12–15**. The first cell checks the source artifacts and restores the existing execution and claim evidence. Cell 15 writes `stage_08_blind_reviewer_packets_v3.zip` under `/kaggle/working/stage08_full_review_v3/review_packets`. Reviewer 1 receives 756 claims and Reviewer 2 receives a frozen independent sample of 300. Share each person's own template, the three context files, and the review instructions. Keep their answers separate.
3. Save the **new** `/kaggle/working/stage08_full_review_v3/private/blinding_key.bin` privately. Keep the notebook's working directory and key out of public outputs. If the session resets, restore that exact key at the same path or set `STAGE08_BLINDING_KEY_FILE` to its private local path before Cell 13. Then rerun from Cell 12 to recreate the same IDs and sample. Use the same key throughout this review round.
4. Attach the corrected round's `reviewer_1_completed.csv` and `reviewer_2_completed.csv` as Kaggle inputs. Run **Cells 16–18**. Keep one discoverable file per role or set `STAGE08_REVIEWER1_FILE` and `STAGE08_REVIEWER2_FILE` to the exact files.
5. If Cell 18 reports disagreements, share `stage_08_adjudication_packet_v3.zip` with Reviewer 3. Attach the returned `reviewer_3_completed.csv` and run **Cells 19–22**. If there are no disagreements, run Cells 19–22 directly.
6. Publish the files in `/kaggle/working/stage08_full_review_v3/stage_08_fresh_dataset` as a **new Stage 08 dataset version** after Cell 22 prints `CHECKPOINT C COMPLETE`. That release is the Notebook 09 input. Preserve the original release for provenance. Notebook 09 still applies the missing-evidence rules for evaluator-unavailable results.

The corrected round uses new blind IDs. Use the v3 templates and keep the nine-claim review with the original release records. Earlier execution, fault replay and model-generation cells are represented by the verified evidence loaded in Cell 12.


In [1]:
# Notebook 08 · Cell 12 · Verify and restore frozen pilot evidence

from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
import csv
import hashlib
import io
import json
import os
import re
import zipfile

STAGE08_VERSION = "stage08-fresh-pilot-v3-full-claim-review"
SOURCE_ROOT = Path(os.environ.get("STAGE08_SOURCE_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("STAGE08_REVIEW_WORK_ROOT", "/kaggle/working/stage08_full_review_v3"))
PROJECT_ROOT = WORK_ROOT / "project"
DATASET_ROOT = WORK_ROOT / "stage_08_fresh_dataset"
CHECKPOINT_ROOT = WORK_ROOT / "review_packets"
for directory in (PROJECT_ROOT, DATASET_ROOT, CHECKPOINT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

# The four external manifests identify the exact releases audited for this correction.
SOURCE_MANIFESTS = {
    "05": "14698c5c34d1b693d062f993bb71d48b9f9b73d57d16c07613519dbda111ad0d",
    "06": "d47e519366bfe44ea9364f9539da838d221aff686215ddeb141eeb84f462b902",
    "07": "3da3c1a7cca4cdd344d6cbaa5f1564632dc453cb937fcd8e38ff935c1836b1bb",
    "08": "a2f249f8294b62b86e28666c4a8a0f4c129bcd3ae0bf3387fd8958524904a261",
}


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_bytes(value):
    return (json.dumps(value, sort_keys=True, ensure_ascii=False,
                       separators=(",", ":")) + "\n").encode("utf-8")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()
            if line.strip()]


def safe_relative(name):
    path = Path(name)
    if not name or path.is_absolute() or ".." in path.parts or str(path) != name:
        raise RuntimeError("Unsafe registered path: " + str(name))
    return path


def immutable_write(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file():
        if path.read_bytes() != data:
            raise RuntimeError("Immutable artifact changed: " + str(path))
        return "REUSED_IDENTICAL"
    if path.exists():
        raise RuntimeError("Non-file artifact at destination: " + str(path))
    temporary = path.with_name(path.name + ".tmp-" + str(os.getpid()))
    try:
        with temporary.open("xb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        if path.is_file() and path.read_bytes() != data:
            raise RuntimeError("Concurrent immutable artifact conflict")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    return "WRITTEN"


def atomic_write(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp-" + str(os.getpid()))
    try:
        with temporary.open("wb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def immutable_json(path, value):
    return immutable_write(path, json_bytes(value))


def atomic_json(path, value):
    return atomic_write(path, json_bytes(value))


def write_jsonl(path, rows, immutable=True):
    payload = b"".join(json_bytes(row) for row in rows)
    return immutable_write(path, payload) if immutable else atomic_write(path, payload)


def write_csv(path, fields, records, immutable=True):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fields, lineterminator="\n",
                            extrasaction="raise")
    writer.writeheader()
    writer.writerows(records)
    payload = buffer.getvalue().encode("utf-8")
    return immutable_write(path, payload) if immutable else atomic_write(path, payload)


def report_cell(number, **data):
    report = {"notebook": "08", "cell": number, "status": "PASS",
              "version": STAGE08_VERSION, **data}
    atomic_json(PROJECT_ROOT / "reports/audit" / f"cell{number:02d}.json", report)
    print(json.dumps(report, indent=2))
    print(f"\nNOTEBOOK 08 — CELL {number:02d}: PASS")
    return report


def require_pass(number):
    path = PROJECT_ROOT / "reports/audit" / f"cell{number:02d}.json"
    if not path.is_file() or read_json(path).get("status") != "PASS":
        raise RuntimeError(f"Required Notebook 08 Cell {number:02d} has not passed")



ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in PRODUCER_NOTEBOOKS or key >= stage or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)



# Restore only the registered execution and extraction evidence from the earlier
# Stage 08. Completed review files and completion declarations belong to v2.
def verified_release(stage, manifest_hash):
    matching = [p for p in SOURCE_ROOT.rglob(f"stage_{stage}_manifest.json")
                if p.is_file() and sha256_file(p) == manifest_hash]
    if len(matching) != 1:
        raise RuntimeError(f"Attach exactly one verified Stage {stage} input; found {len(matching)} matching manifests")
    external_path = matching[0]
    external = read_json(external_path)
    if (external["stage"], external["status"], external["design_id"]) != (
            stage, "PASS", "openapi-fresh-pilot-v1"):
        raise RuntimeError(f"Stage {stage} release identity differs")
    relative = f"registries/{stage}_manifest.json"
    archive_path = external_path.parent / f"stage_{stage}_fresh.zip"
    if archive_path.is_file():
        if sha256_file(archive_path) != external["archive_sha256"]:
            raise RuntimeError(f"Stage {stage} archive hash differs")
        with zipfile.ZipFile(archive_path) as archive:
            names = archive.namelist()
            manifests = [n for n in names if n == relative or n.endswith("/" + relative)]
            if len(set(names)) != len(names) or len(manifests) != 1:
                raise RuntimeError(f"Stage {stage} archive layout differs")
            prefix = manifests[0][:-len(relative)]
            manifest_bytes = archive.read(manifests[0])
            internal = json.loads(manifest_bytes)
            verify_artifact_provenance(internal)
            payloads = {artifact_path_for(row): archive.read(prefix + artifact_path_for(row))
                        for row in internal["artifacts"]}
    else:
        candidates = [p for p in external_path.parent.rglob(relative) if p.is_file()
                      and sha256_file(p) == external["internal_manifest_sha256"]]
        if len(candidates) != 1:
            raise RuntimeError(f"Stage {stage} expanded input has {len(candidates)} matching internal manifests")
        path = candidates[0]
        root = path.parent.parent
        manifest_bytes = path.read_bytes()
        internal = json.loads(manifest_bytes)
        verify_artifact_provenance(internal)
        payloads = {artifact_path_for(row): (root / artifact_path_for(row)).read_bytes()
                    for row in internal["artifacts"]}
    internal_hash = sha256_bytes(manifest_bytes)
    if internal["stage"] != stage or internal_hash != external["internal_manifest_sha256"]:
        raise RuntimeError(f"Stage {stage} internal manifest differs")
    for row in internal["artifacts"]:
        path = artifact_path_for(row)
        data = payloads[path]
        if len(data) != row["bytes"] or sha256_bytes(data) != row["sha256"]:
            raise RuntimeError(f"Stage {stage} artifact differs: {path}")
    return internal, payloads, {"internal_sha256": internal_hash, "external_sha256": manifest_hash}


releases = {stage: verified_release(stage, digest) for stage, digest in SOURCE_MANIFESTS.items()}
stage05, stage06, stage07, stage08 = (releases[stage][1] for stage in ("05", "06", "07", "08"))
SOURCE_IDENTITIES = {stage: releases[stage][2] for stage in ("05", "06", "07")}
if releases["08"][0]["upstream_manifests"] != SOURCE_IDENTITIES:
    raise RuntimeError("The earlier Stage 08 did not consume these Stage 05–07 releases")
if (releases["06"][0]["upstream_manifest_sha256"]["05"] != SOURCE_IDENTITIES["05"]["internal_sha256"]
        or releases["07"][0]["upstream_manifest_sha256"]["06"] != SOURCE_IDENTITIES["06"]["internal_sha256"]):
    raise RuntimeError("Stage 05–07 chain differs")

def payload_jsonl(data):
    return [json.loads(line) for line in data.splitlines() if line.strip()]


def payload_csv(data):
    return list(csv.DictReader(io.StringIO(data.decode("utf-8-sig"))))


TASKS = payload_csv(stage06["data/tasks.csv"])
TASK_BY_ID = {row["task_id"]: row for row in TASKS}
ARTIFACT_BY_TASK = {row["task_id"]: row for row in payload_csv(stage07["data/artifacts.csv"])}
EXECUTION_ROWS = payload_csv(stage08["data/evidence/execution.csv"])
INPUT_INVENTORY = payload_jsonl(stage08["data/inventory/input_artifact_inventory.jsonl"])
OPPORTUNITIES = payload_jsonl(stage08["data/evidence/opportunities_frozen.jsonl"])
RECONCILED_OPPORTUNITIES = payload_jsonl(stage08["data/evidence/opportunity_outcomes.jsonl"])
ASSERTION_CLAIMS = payload_jsonl(stage08["data/evidence/assertion_claims_internal.jsonl"])
BASELINE_ATTEMPTS = payload_jsonl(stage08["data/evidence/request_attempts.jsonl"])

if len(TASK_BY_ID) != len(TASKS) or len(TASKS) != 200 or set(ARTIFACT_BY_TASK) != set(TASK_BY_ID):
    raise RuntimeError("Frozen task or generation frame differs")
for rows in (EXECUTION_ROWS, INPUT_INVENTORY):
    if len(rows) != 200 or {r["task_id"] for r in rows} != set(TASK_BY_ID):
        raise RuntimeError("Frozen execution task accounting differs")
opportunity_ids = {r["opportunity_id"] for r in OPPORTUNITIES}
if (len(opportunity_ids) != 4024 or len(OPPORTUNITIES) != 4024
        or len(RECONCILED_OPPORTUNITIES) != 4024
        or {r["opportunity_id"] for r in RECONCILED_OPPORTUNITIES} != opportunity_ids):
    raise RuntimeError("Frozen opportunity frame differs")
if len(ASSERTION_CLAIMS) != 756 or len({r["claim_id"] for r in ASSERTION_CLAIMS}) != 756:
    raise RuntimeError("Expected the exact 756-claim extraction frame")

for row in INPUT_INVENTORY:
    artifact = ARTIFACT_BY_TASK[row["task_id"]]
    if (row["raw_sha256"] != (artifact["raw_hash"] or None)
            or row["extracted_sha256"] != (artifact["extracted_hash"] or None)
            or row["extraction_status"] != artifact["extraction_status"]):
        raise RuntimeError("Frozen generation inventory differs")
for claim in ASSERTION_CLAIMS:
    task = TASK_BY_ID[claim["task_id"]]
    if any(claim[k] != task[k] for k in ("op_id", "pair_id", "condition")):
        raise RuntimeError("Claim task provenance differs")
    if claim["artifact_id"] != ARTIFACT_BY_TASK[task["task_id"]]["artifact_id"]:
        raise RuntimeError("Claim source provenance differs")

review_evidence = {"data/evidence/assertion_claims.jsonl",
                   "data/evidence/claim_reviews_reconciled.csv",
                   "data/evidence/review_reliability.json",
                   "data/evidence/review_reliability.csv"}
prior_audits = {f"reports/audit/cell{i:02d}.json" for i in range(1, 13)}
for name, data in stage08.items():
    if (name in prior_audits
            or (name.startswith(("config/", "data/evidence/", "data/inventory/",
                                 "reports/qualification/", "src/")) and name not in review_evidence)):
        immutable_write(PROJECT_ROOT / name, data)

immutable_json(PROJECT_ROOT / "config/stage08_review_protocol.json", {
    "review_protocol_version": STAGE08_VERSION,
    "parent_stage08": releases["08"][2],
    "claim_frame_sha256": sha256_bytes(stage08["data/evidence/assertion_claims_internal.jsonl"]),
    "reviewer_1_required": 756, "reviewer_2_required": 300, "review_seed": 20261111,
    "reviewer_2_sampling": "uniform_without_replacement_from_all_extracted_claim_ids",
    "execution_and_claim_extraction_reused": True,
    "reachability_is_recorded_separately_from_semantic_review": True,
})

print("Verified Stage 05–08 inputs, 200 tasks, 4,024 opportunities and 756 extracted claims.")
print("Baseline, variation, fault and claim-extraction evidence restored by hash.")


Verified Stage 05–08 inputs, 200 tasks, 4,024 opportunities and 756 extracted claims.
Baseline, variation, fault and claim-extraction evidence restored by hash.


In [2]:
# Notebook 08 · Cell 13 · Blind all 756 claims

# Use the same review key for every session of this corrected run.
import hmac
import secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

KEY_PATH = WORK_ROOT / "private/blinding_key.bin"
encrypted_path = PROJECT_ROOT / "private/blind_mapping.aesgcm"
key_source = os.environ.get("STAGE08_BLINDING_KEY_FILE", "").strip()
if key_source:
    immutable_write(KEY_PATH, Path(key_source).read_bytes())
if not KEY_PATH.is_file():
    if encrypted_path.exists():
        raise RuntimeError("BLINDING_BLOCK: the corrected run's private key is missing")
    immutable_write(KEY_PATH, secrets.token_bytes(32))
KEY_PATH.chmod(0o600)
BLINDING_KEY = KEY_PATH.read_bytes()
if len(BLINDING_KEY) != 32:
    raise RuntimeError("Expected a 32-byte blinding key")

def blind_identifier(prefix, identity):
    message = f"{STAGE08_VERSION}:{prefix}:{identity}".encode("utf-8")
    return prefix + "_" + hmac.new(BLINDING_KEY, message, hashlib.sha256).hexdigest()[:24]


def blind_case_context(attempt):
    return {key: attempt.get(key) for key in
            ("pytest_item_id", "item_request_ordinal", "status", "scenario_id", "response_status")}


attempts_by_task_and_item = defaultdict(list)
for attempt in BASELINE_ATTEMPTS:
    attempts_by_task_and_item[attempt["task_id"], attempt.get("pytest_item_id")].append(
        blind_case_context(attempt)
    )

BLIND_MAP = {}
BLINDED_CLAIMS = []
BLIND_TEST_CONTEXT = {}
BLIND_SPEC_CONTEXT = {}
REQUIREMENTS_BY_OPERATION = {}
for claim in ASSERTION_CLAIMS:
    op_id = claim["op_id"]
    artifact_id = claim["artifact_id"]
    blind_id = blind_identifier("bc", claim["claim_id"])
    test_context_id = blind_identifier("bt", artifact_id)
    spec_context_id = blind_identifier("bs", op_id)
    if blind_id in BLIND_MAP:
        raise RuntimeError("Blinded claim identity collision")
    BLIND_MAP[blind_id] = {"claim_id": claim["claim_id"], "task_id": claim["task_id"],
                          "op_id": op_id, "condition": claim["condition"],
                          "pair_id": claim["pair_id"]}
    if spec_context_id not in BLIND_SPEC_CONTEXT:
        document = json.loads(stage06[f"data/inputs/L0/{op_id}.json"])
        requirement_rows = payload_jsonl(
            stage05[f"oracles/frozen/pilot/{op_id}/requirements.jsonl"]
        )
        REQUIREMENTS_BY_OPERATION[op_id] = {r["requirement_id"]: r for r in requirement_rows}
        BLIND_SPEC_CONTEXT[spec_context_id] = {
            "spec_context_id": spec_context_id, "openapi_document": document
        }
    if test_context_id not in BLIND_TEST_CONTEXT:
        source_bytes = stage07[f"extracted/{artifact_id}/test_generated.py"]
        if sha256_bytes(source_bytes) != ARTIFACT_BY_TASK[claim["task_id"]]["extracted_hash"]:
            raise RuntimeError("Frozen test source differs from the generation ledger")
        BLIND_TEST_CONTEXT[test_context_id] = {
            "test_context_id": test_context_id, "source": source_bytes.decode("utf-8")
        }
    requirements = REQUIREMENTS_BY_OPERATION[op_id]
    evidence = []
    for requirement_id in claim["requirement_candidates"]:
        requirement = requirements[requirement_id]
        evidence.append({key: requirement.get(key) for key in
                         ("requirement_id", "kind", "expected_rule", "source_pointer", "evidence_sha256")})
    BLINDED_CLAIMS.append({
        "blind_claim_id": blind_id,
        "normalized_claim": claim["normalized_claim"], "source_span": claim["source_span"],
        "line": claim["line"], "column": claim["column"],
        "test_context_id": test_context_id, "spec_context_id": spec_context_id,
        "reachability": claim["reachability"],
        "observed_scenario_ids": claim["observed_scenario_ids"],
        "reached_case_evidence": [
            {"pytest_item_id": item_id,
             "baseline_request_attempts": attempts_by_task_and_item[claim["task_id"], item_id]}
            for item_id in claim["reached_item_ids"]
        ],
        "candidate_evidence": evidence,
    })
BLINDED_CLAIMS.sort(key=lambda row: row["blind_claim_id"])

associated_data = json_bytes(read_json(PROJECT_ROOT / "config/stage08_review_protocol.json"))
if encrypted_path.is_file():
    payload = encrypted_path.read_bytes()
    restored = AESGCM(BLINDING_KEY).decrypt(payload[:12], payload[12:], associated_data)
    if json.loads(restored) != BLIND_MAP:
        raise RuntimeError("BLINDING_BLOCK: mapping differs from the frozen review frame")
else:
    nonce = secrets.token_bytes(12)
    immutable_write(encrypted_path, nonce + AESGCM(BLINDING_KEY).encrypt(
        nonce, json_bytes(BLIND_MAP), associated_data))

OUTGOING = PROJECT_ROOT / "reviews/outgoing"
write_jsonl(OUTGOING / "context_bundle.jsonl", BLINDED_CLAIMS)
write_jsonl(OUTGOING / "test_context.jsonl",
            [BLIND_TEST_CONTEXT[k] for k in sorted(BLIND_TEST_CONTEXT)])
write_jsonl(OUTGOING / "specification_context.jsonl",
            [BLIND_SPEC_CONTEXT[k] for k in sorted(BLIND_SPEC_CONTEXT)])
CONTEXT_FILES = ("context_bundle.jsonl", "test_context.jsonl", "specification_context.jsonl")
immutable_json(OUTGOING / "blind_packet_manifest.json", {
    "status": "PASS", "review_protocol_version": STAGE08_VERSION,
    "claim_count": len(BLINDED_CLAIMS),
    "context_file_sha256": {name: sha256_file(OUTGOING / name) for name in CONTEXT_FILES},
})
report_cell(13, blinded_claims=len(BLINDED_CLAIMS),
            reachability_counts=dict(Counter(row["reachability"] for row in BLINDED_CLAIMS)))
print("Keep the corrected run's key private:", KEY_PATH)


{
  "notebook": "08",
  "cell": 13,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "blinded_claims": 756,
  "reachability_counts": {
    "NOT_REACHED": 740,
    "REACHED": 9,
    "UNVERIFIED": 7
  }
}

NOTEBOOK 08 — CELL 13: PASS
Keep the corrected run's key private: /kaggle/working/stage08_full_review_v3/private/blinding_key.bin


In [3]:
# Notebook 08 · Cell 14 · Freeze Reviewer 2’s seeded sample

from __future__ import annotations

import json
import random
import time


require_pass(13)
REVIEW_SEED = 20261111
ALL_BLIND_IDS = sorted(row["blind_claim_id"] for row in BLINDED_CLAIMS)
if len(set(ALL_BLIND_IDS)) != len(ALL_BLIND_IDS):
    raise RuntimeError("Duplicate blinded review identity")
REVIEW2_COUNT = min(len(ALL_BLIND_IDS), 300)
REVIEW2_IDS = sorted(random.Random(REVIEW_SEED).sample(ALL_BLIND_IDS, REVIEW2_COUNT))
REVIEW2_SAMPLE = {"status": "PASS", "seed": REVIEW_SEED,
                  "population_size": len(ALL_BLIND_IDS), "sample_size": REVIEW2_COUNT,
                  "sampled_ids": REVIEW2_IDS, "sampling": "uniform_without_replacement"}
immutable_json(PROJECT_ROOT / "reviews/outgoing/reviewer_2_frozen_sample.json",
               REVIEW2_SAMPLE)
report_cell(14, total_reviewable_claims=len(ALL_BLIND_IDS),
            reviewer2_sample=REVIEW2_COUNT, sample_seed=REVIEW_SEED)


{
  "notebook": "08",
  "cell": 14,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "total_reviewable_claims": 756,
  "reviewer2_sample": 300,
  "sample_seed": 20261111
}

NOTEBOOK 08 — CELL 14: PASS


{'notebook': '08',
 'cell': 14,
 'status': 'PASS',
 'version': 'stage08-fresh-pilot-v3-full-claim-review',
 'total_reviewable_claims': 756,
 'reviewer2_sample': 300,
 'sample_seed': 20261111}

In [4]:
# Notebook 08 · Cell 15 · Export both independent review assignments

from __future__ import annotations

import json
import os
import time
import zipfile


require_pass(14)
REVIEW_COLUMNS = ("blind_claim_id", "reviewer_id", "label", "requirement_id",
                  "evidence_pointer", "reasoning", "adjudication_status")
ALLOWED_LABELS = ("supported", "contradicted", "undocumented_or_unsupported",
                  "ambiguous_or_unverifiable")


def review_template(blind_ids, reviewer_id):
    return [{"blind_claim_id": blind_id, "reviewer_id": reviewer_id,
             "label": "", "requirement_id": "", "evidence_pointer": "",
             "reasoning": "", "adjudication_status": "NOT_REQUIRED"}
            for blind_id in blind_ids]


write_csv(PROJECT_ROOT / "reviews/outgoing/reviewer_1_template.csv",
          REVIEW_COLUMNS, review_template(ALL_BLIND_IDS, "reviewer_1"))
write_csv(PROJECT_ROOT / "reviews/outgoing/reviewer_2_template.csv",
          REVIEW_COLUMNS, review_template(REVIEW2_IDS, "reviewer_2"))

REVIEW_GATE = {"evaluation_version": STAGE08_VERSION,
               "source_identities": SOURCE_IDENTITIES,
               "status": "AWAITING_REVIEWER_1_AND_2",
               "reviewer_1_required": len(ALL_BLIND_IDS),
               "reviewer_2_required": len(REVIEW2_IDS),
               "reviewer_3_required": "ONLY_IF_DISAGREEMENTS",
               "unblinding_authorized": False,
               "completed_stage08_dataset_authorized": False}
immutable_json(PROJECT_ROOT / "reports/checkpoint_c_review_gate.json", REVIEW_GATE)

REVIEW_INSTRUCTIONS = '# Independent assertion-claim review\n\nReviewer 1 labels all assigned claims. Reviewer 2 labels only the frozen sample, independently. Reviewer 3 adjudicates only label disagreements using the two prior decisions and the blinded evidence.\n\nFor each blind_claim_id, find the matching row in context_bundle.jsonl. Read the assertion and the source location. Use test_context_id to look up the test in test_context.jsonl and spec_context_id to look up the OpenAPI specification in specification_context.jsonl. Each JSONL line contains one JSON object. Check the applicable scenario, test inputs and expected value. For reached assertions, use reached_case_evidence to join the pytest case to its recorded scenario and response. For other assertions, judge the documented contract against the scenario inferable from the test and record any uncertainty. Reachability is recorded independently from the semantic label.\n\n- supported — the applicable specification justifies the expectation.\n- contradicted — applicable evidence conflicts with the expectation.\n- undocumented_or_unsupported — the expectation is clear but the applicable specification does not establish it.\n- ambiguous_or_unverifiable — a necessary value, scenario, or rule cannot be resolved from the evidence.\n\nFill in label and reasoning for every assigned row. For supported and contradicted, copy requirement_id and source_pointer from the same candidate evidence entry, using evidence_pointer for the source pointer. For the other labels, leave both fields blank or cite a matching pair. Keep the headers, blind IDs and reviewer ID intact. Reviewer 1 and 2 keep NOT_REQUIRED. Reviewer 3 marks every final decision COMPLETED, including ambiguous decisions. Return the CSV as reviewer_1_completed.csv, reviewer_2_completed.csv or reviewer_3_completed.csv, according to your assigned role.\n'

immutable_write(OUTGOING / "REVIEW_INSTRUCTIONS.md", REVIEW_INSTRUCTIONS.encode("utf-8"))


def write_packet(filename, payloads):
    path = CHECKPOINT_ROOT / filename
    if not path.is_file():
        temporary = path.with_suffix(".tmp")
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for name, data in payloads.items():
                archive.writestr(name, data)
        os.replace(temporary, path)
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        if (len(names) != len(payloads) or set(names) != set(payloads)
                or any(archive.read(name) != data for name, data in payloads.items())):
            raise RuntimeError(f"Review packet differs from the frozen files: {path}")
    return path


packet_files = (*CONTEXT_FILES, "REVIEW_INSTRUCTIONS.md", "blind_packet_manifest.json",
                "reviewer_1_template.csv", "reviewer_2_template.csv", "reviewer_2_frozen_sample.json")
packet = write_packet("stage_08_blind_reviewer_packets_v3.zip",
                      {name: (OUTGOING / name).read_bytes() for name in packet_files})
report_cell(15, checkpoint_c="AWAITING_REVIEWER_1_AND_2",
            reviewer1_rows=len(ALL_BLIND_IDS), reviewer2_rows=len(REVIEW2_IDS),
            reviewer_packet=str(packet))
print("CHECKPOINT C: Pause here and obtain the two independent reviews.")
print("Share the reviewer packet:", packet)


{
  "notebook": "08",
  "cell": 15,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "checkpoint_c": "AWAITING_REVIEWER_1_AND_2",
  "reviewer1_rows": 756,
  "reviewer2_rows": 300,
  "reviewer_packet": "/kaggle/working/stage08_full_review_v3/review_packets/stage_08_blind_reviewer_packets_v3.zip"
}

NOTEBOOK 08 — CELL 15: PASS
CHECKPOINT C: Pause here and obtain the two independent reviews.
Share the reviewer packet: /kaggle/working/stage08_full_review_v3/review_packets/stage_08_blind_reviewer_packets_v3.zip


In [5]:
# Notebook 08 · Cell 16 · Validate Reviewer 1’s full review

from __future__ import annotations

import csv
import json
import os


require_pass(15)


def discover_review_file(environment_name, expected_name):
    override = os.environ.get(environment_name, "").strip()
    if override:
        path = Path(override)
        if not path.is_file():
            raise RuntimeError("REVIEW_BLOCK: configured review file missing: " + override)
        return path
    matches = sorted(p for p in SOURCE_ROOT.rglob(expected_name)
                     if p.is_file() and not p.is_symlink())
    if len(matches) != 1:
        raise RuntimeError("REVIEW_BLOCK: expected one " + expected_name
                           + f" on Kaggle inputs, found {len(matches)}")
    return matches[0]


def validated_reviews(path, expected_ids, reviewer_id):
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if tuple(reader.fieldnames or []) != REVIEW_COLUMNS:
            raise RuntimeError("REVIEW_BLOCK: review columns differ from frozen schema")
        rows = list(reader)
    if any(None in row or None in row.values() for row in rows):
        raise RuntimeError("REVIEW_BLOCK: CSV row width differs from the frozen schema")
    if (len(rows) != len(expected_ids)
            or len({r["blind_claim_id"] for r in rows}) != len(rows)
            or {r["blind_claim_id"] for r in rows} != set(expected_ids)):
        raise RuntimeError("REVIEW_BLOCK: reviewer identities/count differ from frozen frame")
    evidence_lookup = {r["blind_claim_id"]: r for r in BLINDED_CLAIMS}
    for row in rows:
        if row["reviewer_id"] != reviewer_id or row["label"] not in ALLOWED_LABELS:
            raise RuntimeError("REVIEW_BLOCK: illegal reviewer ID or label")
        if not row["reasoning"].strip():
            raise RuntimeError("REVIEW_BLOCK: reasoning cannot be omitted")
        if row["adjudication_status"] not in ("NOT_REQUIRED", "COMPLETED"):
            raise RuntimeError("REVIEW_BLOCK: unknown adjudication status")
        allowed = {x["requirement_id"]: x["source_pointer"]
                   for x in evidence_lookup[row["blind_claim_id"]]["candidate_evidence"]}
        if row["label"] in ("supported", "contradicted"):
            if (row["requirement_id"] not in allowed
                    or row["evidence_pointer"] != allowed[row["requirement_id"]]):
                raise RuntimeError("REVIEW_BLOCK: supported/contradicted label needs "
                                   "a matching frozen requirement evidence pointer")
        else:
            if row["requirement_id"] and row["requirement_id"] not in allowed:
                raise RuntimeError("REVIEW_BLOCK: unknown requirement ID")
            if (row["requirement_id"]
                    and row["evidence_pointer"] != allowed[row["requirement_id"]]):
                raise RuntimeError("REVIEW_BLOCK: inaccurate evidence pointer")
    return sorted(rows, key=lambda r: r["blind_claim_id"])


R1_SOURCE = discover_review_file("STAGE08_REVIEWER1_FILE", "reviewer_1_completed.csv")
REVIEW1_ROWS = validated_reviews(R1_SOURCE, ALL_BLIND_IDS, "reviewer_1")
write_csv(PROJECT_ROOT / "reviews/completed/reviewer_1.csv", REVIEW_COLUMNS, REVIEW1_ROWS)
report_cell(16, reviewer_1_rows=len(REVIEW1_ROWS),
            csv_sha256=sha256_file(PROJECT_ROOT / "reviews/completed/reviewer_1.csv"))


{
  "notebook": "08",
  "cell": 16,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "reviewer_1_rows": 756,
  "csv_sha256": "a001cea2227182ddf9273025fe62e89ce3c8319bbb0ebd2dbf9feeefc8b4823a"
}

NOTEBOOK 08 — CELL 16: PASS


{'notebook': '08',
 'cell': 16,
 'status': 'PASS',
 'version': 'stage08-fresh-pilot-v3-full-claim-review',
 'reviewer_1_rows': 756,
 'csv_sha256': 'a001cea2227182ddf9273025fe62e89ce3c8319bbb0ebd2dbf9feeefc8b4823a'}

In [6]:
# Notebook 08 · Cell 17 · Validate Reviewer 2’s independent sample

from __future__ import annotations

import json
import time


require_pass(16)
R2_SOURCE = discover_review_file("STAGE08_REVIEWER2_FILE", "reviewer_2_completed.csv")
REVIEW2_ROWS = validated_reviews(R2_SOURCE, REVIEW2_IDS, "reviewer_2")
write_csv(PROJECT_ROOT / "reviews/completed/reviewer_2.csv", REVIEW_COLUMNS, REVIEW2_ROWS)
if {r["reviewer_id"] for r in REVIEW1_ROWS} & {
    r["reviewer_id"] for r in REVIEW2_ROWS
}:
    raise RuntimeError("REVIEW_BLOCK: independent reviewer roles must be distinct")
report_cell(17, reviewer_2_rows=len(REVIEW2_ROWS),
            reviewer_2_frozen_sample_verified=True,
            csv_sha256=sha256_file(PROJECT_ROOT / "reviews/completed/reviewer_2.csv"))


{
  "notebook": "08",
  "cell": 17,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "reviewer_2_rows": 300,
  "reviewer_2_frozen_sample_verified": true,
  "csv_sha256": "3a9a4c4ec696a74f78e3ace43817ece0588c5d7a9b6a96c727bd08d3f6954dfa"
}

NOTEBOOK 08 — CELL 17: PASS


{'notebook': '08',
 'cell': 17,
 'status': 'PASS',
 'version': 'stage08-fresh-pilot-v3-full-claim-review',
 'reviewer_2_rows': 300,
 'reviewer_2_frozen_sample_verified': True,
 'csv_sha256': '3a9a4c4ec696a74f78e3ace43817ece0588c5d7a9b6a96c727bd08d3f6954dfa'}

In [7]:
# Notebook 08 · Cell 18 · Export the disagreement-only adjudication packet

from __future__ import annotations

import json
import zipfile


require_pass(17)
R1_LOOKUP = {r["blind_claim_id"]: r for r in REVIEW1_ROWS}
R2_LOOKUP = {r["blind_claim_id"]: r for r in REVIEW2_ROWS}
DISAGREEMENT_IDS = sorted(
    blind_id for blind_id in REVIEW2_IDS
    if R1_LOOKUP[blind_id]["label"] != R2_LOOKUP[blind_id]["label"]
)
write_csv(PROJECT_ROOT / "reviews/outgoing/reviewer_3_template.csv",
          REVIEW_COLUMNS, review_template(DISAGREEMENT_IDS, "reviewer_3"))
immutable_json(PROJECT_ROOT / "reviews/outgoing/adjudication_manifest.json", {
    "status": "AWAITING_ADJUDICATION" if DISAGREEMENT_IDS else "NO_ADJUDICATION_NEEDED",
    "reviewer_1_sha256": sha256_file(PROJECT_ROOT / "reviews/completed/reviewer_1.csv"),
    "reviewer_2_sha256": sha256_file(PROJECT_ROOT / "reviews/completed/reviewer_2.csv"),
    "disagreement_count": len(DISAGREEMENT_IDS),
    "disagreement_ids": DISAGREEMENT_IDS,
    "unblinding_authorized": False,
})

if DISAGREEMENT_IDS:
    disagreement_set = set(DISAGREEMENT_IDS)
    contexts = [r for r in BLINDED_CLAIMS if r["blind_claim_id"] in disagreement_set]
    test_ids = sorted({r["test_context_id"] for r in contexts})
    spec_ids = sorted({r["spec_context_id"] for r in contexts})
    packet = write_packet("stage_08_adjudication_packet_v3.zip", {
        "reviewer_3_template.csv": (OUTGOING / "reviewer_3_template.csv").read_bytes(),
        "REVIEW_INSTRUCTIONS.md": (OUTGOING / "REVIEW_INSTRUCTIONS.md").read_bytes(),
        "context_bundle.jsonl": b"".join(map(json_bytes, contexts)),
        "test_context.jsonl": b"".join(json_bytes(BLIND_TEST_CONTEXT[k]) for k in test_ids),
        "specification_context.jsonl": b"".join(json_bytes(BLIND_SPEC_CONTEXT[k]) for k in spec_ids),
        "prior_decisions.jsonl": b"".join(json_bytes({
            "blind_claim_id": k, "reviewer_1": R1_LOOKUP[k], "reviewer_2": R2_LOOKUP[k]
        }) for k in DISAGREEMENT_IDS),
    })
    print("Pause for one adjudication per disagreement. Share:", packet)
else:
    print("The two reviewers agree on every sampled claim. Continue to Cell 19.")

report_cell(18, reviewer_2_overlap=len(REVIEW2_IDS),
            disagreements=len(DISAGREEMENT_IDS),
            checkpoint_c=("AWAITING_ADJUDICATION" if DISAGREEMENT_IDS
                          else "NO_ADJUDICATION_NEEDED"))


Pause for one adjudication per disagreement. Share: /kaggle/working/stage08_full_review_v3/review_packets/stage_08_adjudication_packet_v3.zip
{
  "notebook": "08",
  "cell": 18,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "reviewer_2_overlap": 300,
  "disagreements": 72,
  "checkpoint_c": "AWAITING_ADJUDICATION"
}

NOTEBOOK 08 — CELL 18: PASS


{'notebook': '08',
 'cell': 18,
 'status': 'PASS',
 'version': 'stage08-fresh-pilot-v3-full-claim-review',
 'reviewer_2_overlap': 300,
 'disagreements': 72,
 'checkpoint_c': 'AWAITING_ADJUDICATION'}

In [8]:
# Notebook 08 · Cell 19 · Validate adjudication and reconcile all claims

from __future__ import annotations

import csv
import json


require_pass(18)
if DISAGREEMENT_IDS:
    R3_SOURCE = discover_review_file("STAGE08_REVIEWER3_FILE",
                                     "reviewer_3_completed.csv")
    REVIEW3_ROWS = validated_reviews(R3_SOURCE, DISAGREEMENT_IDS, "reviewer_3")
    if any(row["adjudication_status"] != "COMPLETED" for row in REVIEW3_ROWS):
        raise RuntimeError("ADJUDICATION_BLOCK: Reviewer 3 must mark each decision COMPLETED")
else:
    REVIEW3_ROWS = []
write_csv(PROJECT_ROOT / "reviews/completed/reviewer_3.csv", REVIEW_COLUMNS, REVIEW3_ROWS)
R3_LOOKUP = {r["blind_claim_id"]: r for r in REVIEW3_ROWS}

RECONCILED_REVIEWS = []
for blind_id in ALL_BLIND_IDS:
    original = R1_LOOKUP[blind_id]
    second = R2_LOOKUP.get(blind_id)
    third = R3_LOOKUP.get(blind_id)
    if blind_id in DISAGREEMENT_IDS and third is None:
        raise RuntimeError("ADJUDICATION_BLOCK: missing Reviewer 3 decision")
    if second is not None and second["label"] != original["label"]:
        chosen = third
        status = "ADJUDICATED"
    else:
        chosen = original
        status = "AGREEMENT" if second is not None else "REVIEWER_1_ONLY"
    RECONCILED_REVIEWS.append({
        "blind_claim_id": blind_id,
        "reviewer_1_label": original["label"],
        "reviewer_2_label": second["label"] if second else "",
        "reviewer_3_label": third["label"] if third else "",
        "reconciled_label": chosen["label"],
        "requirement_id": chosen["requirement_id"],
        "evidence_pointer": chosen["evidence_pointer"],
        "reasoning": chosen["reasoning"],
        "adjudication_status": status,
    })
RECONCILED_COLUMNS = list(RECONCILED_REVIEWS[0]) if RECONCILED_REVIEWS else [
    "blind_claim_id", "reviewer_1_label", "reviewer_2_label", "reviewer_3_label",
    "reconciled_label", "requirement_id", "evidence_pointer", "reasoning",
    "adjudication_status"]
write_csv(PROJECT_ROOT / "data/evidence/claim_reviews_reconciled.csv",
          RECONCILED_COLUMNS, RECONCILED_REVIEWS)
report_cell(19, total_reviewed_claims=len(RECONCILED_REVIEWS),
            adjudicated_disagreements=len(DISAGREEMENT_IDS),
            original_review_labels_preserved=True)


{
  "notebook": "08",
  "cell": 19,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "total_reviewed_claims": 756,
  "adjudicated_disagreements": 72,
  "original_review_labels_preserved": true
}

NOTEBOOK 08 — CELL 19: PASS


{'notebook': '08',
 'cell': 19,
 'status': 'PASS',
 'version': 'stage08-fresh-pilot-v3-full-claim-review',
 'total_reviewed_claims': 756,
 'adjudicated_disagreements': 72,
 'original_review_labels_preserved': True}

In [9]:
# Notebook 08 · Cell 20 · Report independent reviewer agreement

from __future__ import annotations

from collections import Counter
import json
import math


require_pass(19)
OVERLAP_COUNT = len(REVIEW2_IDS)
matrix = {a: {b: 0 for b in ALLOWED_LABELS} for a in ALLOWED_LABELS}
for blind_id in REVIEW2_IDS:
    matrix[R1_LOOKUP[blind_id]["label"]][R2_LOOKUP[blind_id]["label"]] += 1
agree = sum(matrix[x][x] for x in ALLOWED_LABELS)
observed = agree / OVERLAP_COUNT if OVERLAP_COUNT else None
kappa = None
reliability_reason = None
if OVERLAP_COUNT:
    first_marginal = {x: sum(matrix[x].values()) for x in ALLOWED_LABELS}
    second_marginal = {x: sum(matrix[y][x] for y in ALLOWED_LABELS)
                       for x in ALLOWED_LABELS}
    expected = sum(first_marginal[x] * second_marginal[x] for x in ALLOWED_LABELS)
    expected /= OVERLAP_COUNT ** 2
    if math.isclose(expected, 1.0):
        reliability_reason = "undefined: expected agreement is exactly one"
    else:
        kappa = (observed - expected) / (1.0 - expected)
else:
    reliability_reason = "undefined: no independently reviewed overlap"

# Wilson interval for the raw exact-agreement fraction, not a kappa interval.
wilson_low = wilson_high = None
if OVERLAP_COUNT:
    z = 1.959963984540054
    center = (observed + z*z/(2*OVERLAP_COUNT)) / (1 + z*z/OVERLAP_COUNT)
    spread = z*math.sqrt(observed*(1-observed)/OVERLAP_COUNT +
                         z*z/(4*OVERLAP_COUNT**2)) / (1 + z*z/OVERLAP_COUNT)
    wilson_low = max(0, center-spread)
    wilson_high = min(1, center+spread)

RELIABILITY = {"status": "PASS", "reviewer1_total": len(ALL_BLIND_IDS),
               "overlap_count": OVERLAP_COUNT, "exact_agreements": agree,
               "exact_agreement": observed, "cohens_kappa": kappa,
               "undefined_reliability_reason": reliability_reason,
               "wilson_exact_agreement_interval": [wilson_low, wilson_high],
               "confusion_matrix": matrix,
               "adjudicated_count": len(DISAGREEMENT_IDS)}
immutable_json(PROJECT_ROOT / "data/evidence/review_reliability.json", RELIABILITY)
write_csv(PROJECT_ROOT / "data/evidence/review_reliability.csv",
          ["metric", "value"],
          [{"metric": key, "value": json.dumps(value, sort_keys=True)}
           for key, value in RELIABILITY.items()])
report_cell(20, overlap=OVERLAP_COUNT, exact_agreement=observed,
            cohens_kappa=kappa, undefined_reason=reliability_reason)


{
  "notebook": "08",
  "cell": 20,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "overlap": 300,
  "exact_agreement": 0.76,
  "cohens_kappa": 0.5954222779973403,
  "undefined_reason": null
}

NOTEBOOK 08 — CELL 20: PASS


{'notebook': '08',
 'cell': 20,
 'status': 'PASS',
 'version': 'stage08-fresh-pilot-v3-full-claim-review',
 'overlap': 300,
 'exact_agreement': 0.76,
 'cohens_kappa': 0.5954222779973403,
 'undefined_reason': None}

In [10]:
# Notebook 08 · Cell 21 · Verify the full review and seal Checkpoint C

from __future__ import annotations

from collections import Counter
import json
import time


require_pass(20)
def check_review_ids(rows, expected_ids, role):
    actual_ids = [row["blind_claim_id"] for row in rows]
    if len(actual_ids) != len(expected_ids) or set(actual_ids) != set(expected_ids):
        raise RuntimeError(f"CLAIM_REVIEW_BLOCK: {role} IDs differ from the frozen assignment")


claim_ids = {row["claim_id"] for row in ASSERTION_CLAIMS}
if (len(ALL_BLIND_IDS) != len(claim_ids) or set(ALL_BLIND_IDS) != set(BLIND_MAP)
        or {row["claim_id"] for row in BLIND_MAP.values()} != claim_ids):
    raise RuntimeError("CLAIM_REVIEW_BLOCK: blind frame differs from the full extraction ledger")
frozen_sample = read_json(OUTGOING / "reviewer_2_frozen_sample.json")
if (frozen_sample["sampled_ids"] != REVIEW2_IDS or len(REVIEW2_IDS) != min(len(claim_ids), 300)
        or len(set(REVIEW2_IDS)) != len(REVIEW2_IDS) or not set(REVIEW2_IDS) <= set(ALL_BLIND_IDS)):
    raise RuntimeError("CLAIM_REVIEW_BLOCK: independent sample differs from the frozen assignment")
check_review_ids(REVIEW1_ROWS, ALL_BLIND_IDS, "Reviewer 1")
check_review_ids(REVIEW2_ROWS, REVIEW2_IDS, "Reviewer 2")
expected_disagreements = [k for k in REVIEW2_IDS if R1_LOOKUP[k]["label"] != R2_LOOKUP[k]["label"]]
if sorted(expected_disagreements) != DISAGREEMENT_IDS:
    raise RuntimeError("CLAIM_REVIEW_BLOCK: adjudication scope differs from the label disagreements")
check_review_ids(REVIEW3_ROWS, DISAGREEMENT_IDS, "Reviewer 3")
if any(row["adjudication_status"] != "COMPLETED" for row in REVIEW3_ROWS):
    raise RuntimeError("CLAIM_REVIEW_BLOCK: Reviewer 3 has incomplete decisions")
check_review_ids(RECONCILED_REVIEWS, ALL_BLIND_IDS, "reconciliation")

payload = (PROJECT_ROOT / "private/blind_mapping.aesgcm").read_bytes()
DECRYPTED_MAPPING = json.loads(AESGCM(BLINDING_KEY).decrypt(
    payload[:12], payload[12:], associated_data))
if DECRYPTED_MAPPING != BLIND_MAP:
    raise RuntimeError("UNBLINDING_BLOCK: private mapping integrity failed")
blind_by_claim = {row["claim_id"]: blind for blind, row in DECRYPTED_MAPPING.items()}
reviews_by_blind_id = {row["blind_claim_id"]: row for row in RECONCILED_REVIEWS}

FINAL_CLAIMS = []
for claim in ASSERTION_CLAIMS:
    blind_id = blind_by_claim[claim["claim_id"]]
    review = reviews_by_blind_id[blind_id]
    FINAL_CLAIMS.append({**claim, "blind_claim_id": blind_id,
                         "review_status": "RECONCILED",
                         "review_label": review["reconciled_label"],
                         "requirement_id": review["requirement_id"],
                         "review_evidence_pointer": review["evidence_pointer"],
                         "adjudication_status": review["adjudication_status"]})
write_jsonl(PROJECT_ROOT / "data/evidence/assertion_claims.jsonl", FINAL_CLAIMS)
for name in ("reviewer_2_frozen_sample.json", "blind_packet_manifest.json", "adjudication_manifest.json"):
    immutable_write(PROJECT_ROOT / "data/evidence" / name, (OUTGOING / name).read_bytes())

# upstream identities, including artifacts with no executable source.
if len(EXECUTION_ROWS) != 200 or len({r["task_id"] for r in EXECUTION_ROWS}) != 200:
    raise RuntimeError("TASK_ACCOUNTING_BLOCK: 200 tasks not reconciled")
if len(RECONCILED_OPPORTUNITIES) != len(OPPORTUNITIES):
    raise RuntimeError("OPPORTUNITY_ACCOUNTING_BLOCK: incomplete replay frame")
if len(FINAL_CLAIMS) != len(ASSERTION_CLAIMS):
    raise RuntimeError("CLAIM_ACCOUNTING_BLOCK: incomplete claim frame")

FINAL_SUMMARY = {
    "status": "PASS", "stage": "08", "partition": "pilot",
    "checkpoint_c_complete": True, "task_count": 200, "operation_count": 100,
    "pilot_release_scope": "FRESH_PROBABILITY_SAMPLE",
    "weighted_inference_requires_complete_evidence": True,
    "baseline_evaluator_unavailable_count": sum(
        row["baseline_status"] == "EVALUATOR_UNAVAILABLE" for row in EXECUTION_ROWS
    ),
    "extracted_module_count": sum(bool(r["extracted_sha256"]) for r in INPUT_INVENTORY),
    "reconciled_opportunities": len(RECONCILED_OPPORTUNITIES),
    "assertion_claim_count": len(FINAL_CLAIMS),
    "reviewed_assertion_claims": len(RECONCILED_REVIEWS),
    "reviewed_reached_claims": sum(r["reachability"] == "REACHED" for r in FINAL_CLAIMS),
    "review_protocol_version": STAGE08_VERSION,
    "parent_stage08": releases["08"][2],
    "reviewer_2_overlap": len(REVIEW2_IDS),
    "adjudications": len(DISAGREEMENT_IDS),
    "upstream_manifests": SOURCE_IDENTITIES,
    "frozen_worker_sha256": sha256_file(PROJECT_ROOT / "src/worker.py"),
    "model_outputs_repaired": False,
    "live_api_access_permitted": False,
}
immutable_json(PROJECT_ROOT / "reports/audit/final_stage08_reconciliation.json",
               FINAL_SUMMARY)

# Record completion before freezing the package member list. Re-running Cell
# 21 with an intact working directory must not alter the registered population.
report_cell(21, checkpoint_c_complete=True, task_count=200,
            claim_count=len(FINAL_CLAIMS), reviewer_count=3,
            registered_artifacts="SEE_INTERNAL_MANIFEST")

# The package intentionally references frozen Stage 05/06/07 inputs rather
# than duplicating their oracles, raw generations, and source code.
PACKAGE_PATHS = []
for relative_root in ("config", "data/evidence", "data/inventory", "src",
                      "reports/qualification", "reports/audit", "reviews/completed"):
    folder = PROJECT_ROOT / relative_root
    if folder.is_dir():
        PACKAGE_PATHS.extend(file.relative_to(PROJECT_ROOT).as_posix()
                             for file in folder.rglob("*") if file.is_file()
                             and not file.is_symlink()
                             and "__pycache__" not in file.parts)
PACKAGE_PATHS = sorted(set(PACKAGE_PATHS))
PACKAGE_ARTIFACTS = [
    {"relative_path": relative, "bytes": (PROJECT_ROOT / relative).stat().st_size,
     "sha256": sha256_file(PROJECT_ROOT / relative)}
    for relative in PACKAGE_PATHS
]
INTERNAL_MANIFEST = {"stage": "08", "status": "PASS", "partition": "pilot", "design_id": "openapi-fresh-pilot-v1",
                     "checkpoint_c_complete": True, "task_count": 200,
                     "operation_count": 100, "artifact_count": len(PACKAGE_ARTIFACTS),
                     "artifacts": PACKAGE_ARTIFACTS,
                     "upstream_manifests": SOURCE_IDENTITIES,
                     "reached_claims_reviewed": FINAL_SUMMARY["reviewed_reached_claims"],
                     "assertion_claims_reviewed": len(RECONCILED_REVIEWS),
                     "reviewer_2_sample_size": len(REVIEW2_IDS),
                     "review_protocol_version": STAGE08_VERSION,
                     "parent_stage08": releases["08"][2],
                     "pilot_release_scope": FINAL_SUMMARY["pilot_release_scope"],
                     "weighted_inference_requires_complete_evidence": True,
                                      "generated_source_modified": False}
INTERNAL_MANIFEST["upstream_manifest_sha256"] = {
    stage: identity["internal_sha256"] for stage, identity in SOURCE_IDENTITIES.items()
}
register_artifact_provenance(INTERNAL_MANIFEST)
immutable_json(PROJECT_ROOT / "registries/08_manifest.json", INTERNAL_MANIFEST)
print("Frozen Stage 08 registered artifacts:", len(PACKAGE_ARTIFACTS))


{
  "notebook": "08",
  "cell": 21,
  "status": "PASS",
  "version": "stage08-fresh-pilot-v3-full-claim-review",
  "checkpoint_c_complete": true,
  "task_count": 200,
  "claim_count": 756,
  "reviewer_count": 3,
  "registered_artifacts": "SEE_INTERNAL_MANIFEST"
}

NOTEBOOK 08 — CELL 21: PASS
Frozen Stage 08 registered artifacts: 58


In [11]:
# Notebook 08 · Cell 22 · Package and verify the corrected Stage 08 release

from __future__ import annotations

import json
import os
import zipfile


require_pass(21)
ARCHIVE_NAME = "stage_08_fresh.zip"
ARCHIVE_PATH = DATASET_ROOT / ARCHIVE_NAME
ARCHIVE_PREFIX = "stage_08_fresh/prose_operation_experiment/"
manifest_relative = "registries/08_manifest.json"
members = PACKAGE_PATHS + [manifest_relative]

if not ARCHIVE_PATH.is_file():
    temporary = ARCHIVE_PATH.with_name(ARCHIVE_NAME + ".tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=7, allowZip64=True) as archive:
        for relative in members:
            entry = zipfile.ZipInfo(ARCHIVE_PREFIX + relative,
                                    date_time=(2026, 11, 11, 0, 0, 0))
            entry.compress_type = zipfile.ZIP_DEFLATED
            archive.writestr(entry, (PROJECT_ROOT / relative).read_bytes(),
                             compress_type=zipfile.ZIP_DEFLATED, compresslevel=7)
    os.replace(temporary, ARCHIVE_PATH)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    names = archive.namelist()
    if (len(names) != len(members) or len(set(names)) != len(names)
            or set(names) != {ARCHIVE_PREFIX + x for x in members}):
        raise RuntimeError("Stage 08 ZIP content differs from registered evidence")
    if archive.testzip() is not None:
        raise RuntimeError("Stage 08 ZIP failed CRC qualification")
    for item in PACKAGE_ARTIFACTS:
        data = archive.read(ARCHIVE_PREFIX + item["relative_path"])
        if len(data) != item["bytes"] or sha256_bytes(data) != item["sha256"]:
            raise RuntimeError("Stage 08 registered archive artifact changed")
    internal = archive.read(ARCHIVE_PREFIX + manifest_relative)
    if sha256_bytes(internal) != sha256_file(PROJECT_ROOT / manifest_relative):
        raise RuntimeError("Stage 08 internal manifest changed in ZIP")

EXTERNAL_MANIFEST = {"stage": "08", "status": "PASS", "partition": "pilot", "design_id": "openapi-fresh-pilot-v1",
                     "archive_name": ARCHIVE_NAME,
                     "archive_sha256": sha256_file(ARCHIVE_PATH),
                     "archive_bytes": ARCHIVE_PATH.stat().st_size,
                     "archive": {"filename": ARCHIVE_NAME,
                                 "sha256": sha256_file(ARCHIVE_PATH),
                                 "bytes": ARCHIVE_PATH.stat().st_size,
                                 "member_count": len(members),
                                 "project_prefix": ARCHIVE_PREFIX.rstrip("/")},
                     "internal_manifest_sha256": sha256_bytes(internal),
                     "internal_manifest": {"relative_path": manifest_relative,
                                           "sha256": sha256_bytes(internal),
                                           "artifact_count": len(PACKAGE_ARTIFACTS)},
                     "task_count": 200, "operation_count": 100,
                     "checkpoint_c_complete": True,
                     "may_begin_notebook_09": True,
                     "assertion_claims_reviewed": len(RECONCILED_REVIEWS),
                     "reviewer_2_sample_size": len(REVIEW2_IDS),
                     "review_protocol_version": STAGE08_VERSION,
                     "pilot_release_scope": FINAL_SUMMARY["pilot_release_scope"],
                     "weighted_inference_requires_complete_evidence": True,
                     "main_sample_sizing_authorized": False}
external_path = DATASET_ROOT / "stage_08_manifest.json"
immutable_json(external_path, EXTERNAL_MANIFEST)
immutable_write(DATASET_ROOT / "stage_08_checksums.sha256", (
    sha256_file(ARCHIVE_PATH) + "  " + ARCHIVE_NAME + "\n"
    + sha256_file(external_path) + "  stage_08_manifest.json\n").encode("utf-8"))
immutable_write(DATASET_ROOT / "README.md", (
    "# Stage 08 — Pilot Offline Evaluation and Human Review\n\n"
    "Status: PASS; Checkpoint C complete.\n\n"
    "This lean, noncumulative package preserves the full 200-task execution "
    "and replay opportunity accounting, full assertion-claim reviews, reviewer "
    "reliability and the unblinded final evidence used by Notebook 09.\n"
    "Attach frozen Stages 05, 06, 07 and 08 for any provenance replay.\n"
    "Notebook 09 checks score and inference readiness from the completed evidence.\n"
).encode("utf-8"))
immutable_write(PROJECT_ROOT / "stage_summary_for_executed.md", (
    "# Notebook 08 — PASS\n\n"
    "- 200 frozen tasks reconciled.\n"
    f"- {len(RECONCILED_OPPORTUNITIES)} oracle opportunities accounted for.\n"
    f"- {len(RECONCILED_REVIEWS)} assertion claims reviewed.\n"
    f"- {len(DISAGREEMENT_IDS)} adjudicated disagreements.\n"
    "- Checkpoint C complete; Notebook 09 authorized.\n"
).encode("utf-8"))

print(json.dumps({"status": "PASS", "stage": "08", "task_count": 200,
                  "checkpoint_c_complete": True,
                  "archive_sha256": EXTERNAL_MANIFEST["archive_sha256"],
                  "registered_artifacts": len(PACKAGE_ARTIFACTS),
                  "may_begin_notebook_09": True}, indent=2))
print("\nNOTEBOOK 08 — CELL 22: PASS")
print("CHECKPOINT C COMPLETE. Publish Stage 08 and proceed to Notebook 09.")


{
  "status": "PASS",
  "stage": "08",
  "task_count": 200,
  "checkpoint_c_complete": true,
  "archive_sha256": "03ebff585b2f10e2a916ab916961f295179dcd030824a7bf11ee71dc9eb09614",
  "registered_artifacts": 58,
  "may_begin_notebook_09": true
}

NOTEBOOK 08 — CELL 22: PASS
CHECKPOINT C COMPLETE. Publish Stage 08 and proceed to Notebook 09.
